<a href="https://colab.research.google.com/github/gowripreetham/SJSU_Deep_Learning_Advanced-customizations-in-deep-learning-and-neural-networks/blob/main/09_keras_custom_optimizer_and_training_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 09: Keras Custom Optimizer and Custom Training Loop

**Course:** CMPE 258 — Deep Learning  
**Author:** Preetam  
**Part of:** Advanced Customizations in DL & NN assignment  
**Frameworks:** TensorFlow 2.16 / Keras 3.3  
**Companion video:** TBD

## What this notebook covers
- Custom optimizer (`MyMomentumOptimizer`)
- Manual `GradientTape` loop with metrics and tqdm
- `@tf.function` training-step speed comparison

## Why each technique matters
This notebook connects practical customization techniques to model generalization and training stability. Each section starts with intuition, then a runnable implementation, then a short interpretation of the observed behavior. Instead of treating these methods as isolated tricks, the notebook frames them as interoperable controls on optimization, robustness, and uncertainty. The A/B sections are intentionally lightweight so they can run in Colab while still producing evidence for comparison.


In [ ]:
!pip -q install tensorflow==2.16.1 keras==3.3.3 tqdm
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Python: 3.11.9
Platform: macOS-26.0.1-arm64-arm-64bit


In [ ]:
# Set deterministic seeds for reproducibility.
import os
import random
import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
import keras

tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)


Fashion MNIST is used as a stable benchmark for comparing optimizer behavior and manual-vs-compiled training loops.


## Custom optimizer: SGD with momentum


In [ ]:
import time
import keras
import tensorflow as tf
from keras import layers
from tqdm.auto import tqdm
import numpy as np

class MyMomentumOptimizer(keras.optimizers.Optimizer):
    def __init__(self, learning_rate=0.01, momentum=0.9, name="MyMomentumOptimizer", **kwargs):
        super().__init__(name=name, learning_rate=learning_rate, **kwargs)
        self.momentum = momentum
    def build(self, variables):
        super().build(variables)
        self.momentums = []
        for var in variables:
            self.momentums.append(self.add_variable_from_reference(reference_variable=var, name="m"))
    def update_step(self, gradient, variable, learning_rate):
        idx = self._get_variable_index(variable)
        m = self.momentums[idx]
        self.assign(m, self.momentum * m - learning_rate * gradient)
        self.assign_add(variable, m)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({"momentum": self.momentum})
        return cfg


## Manual GradientTape training loop


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = (x_train.astype("float32") / 255.0).reshape(-1, 784)
x_test = (x_test.astype("float32") / 255.0).reshape(-1, 784)

train_ds = tf.data.Dataset.from_tensor_slices((x_train[:12000], y_train[:12000])).shuffle(2048).batch(128)
val_ds = tf.data.Dataset.from_tensor_slices((x_train[12000:15000], y_train[12000:15000])).batch(256)

def make_model():
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])

model_manual = make_model()
opt = MyMomentumOptimizer(learning_rate=0.01, momentum=0.9)
loss_fn = keras.losses.SparseCategoricalCrossentropy()
train_acc = keras.metrics.SparseCategoricalAccuracy()
val_acc = keras.metrics.SparseCategoricalAccuracy()

@tf.function
def train_step_tf(xb, yb):
    with tf.GradientTape() as tape:
        pred = model_manual(xb, training=True)
        loss = loss_fn(yb, pred) + tf.add_n(model_manual.losses)
    grads = tape.gradient(loss, model_manual.trainable_variables)
    opt.apply_gradients(zip(grads, model_manual.trainable_variables))
    train_acc.update_state(yb, pred)
    return loss

def train_step_eager(xb, yb):
    with tf.GradientTape() as tape:
        pred = model_manual(xb, training=True)
        loss = loss_fn(yb, pred) + tf.add_n(model_manual.losses)
    grads = tape.gradient(loss, model_manual.trainable_variables)
    opt.apply_gradients(zip(grads, model_manual.trainable_variables))
    train_acc.update_state(yb, pred)
    return loss

for epoch in range(3):
    train_acc.reset_state()
    pbar = tqdm(train_ds, desc=f"Epoch {epoch+1}")
    for step, (xb, yb) in enumerate(pbar):
        loss = train_step_eager(xb, yb)
        if step % 100 == 0:
            pbar.set_postfix({"loss": float(loss)})
    val_acc.reset_state()
    for xb, yb in val_ds:
        val_acc.update_state(yb, model_manual(xb, training=False))
    print(f"Epoch {epoch+1} train_acc={float(train_acc.result()):.4f} val_acc={float(val_acc.result()):.4f}")


Epoch 1:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 1 train_acc=0.6522 val_acc=0.7707


2026-04-25 13:31:03.058535: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-04-25 13:31:03.090873: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 2:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 2 train_acc=0.7952 val_acc=0.7897


2026-04-25 13:31:03.855051: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-04-25 13:31:03.883763: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 3:   0%|          | 0/94 [00:00<?, ?it/s]

Epoch 3 train_acc=0.8220 val_acc=0.8220


2026-04-25 13:31:04.647432: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-04-25 13:31:04.675243: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Compare custom loop with model.fit baseline and tf.function speedup


In [ ]:
model_fit = make_model()
model_fit.compile(optimizer=keras.optimizers.SGD(0.01, momentum=0.9), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_fit = model_fit.fit(x_train[:12000], y_train[:12000], validation_data=(x_train[12000:15000], y_train[12000:15000]), epochs=3, batch_size=128, verbose=0)
print("model.fit final val_acc:", h_fit.history["val_accuracy"][-1])

xb, yb = next(iter(train_ds))
_ = train_step_tf(xb, yb)  # warmup
t0 = time.time()
for xb, yb in train_ds.take(40):
    _ = train_step_eager(xb, yb)
t1 = time.time()
for xb, yb in train_ds.take(40):
    _ = train_step_tf(xb, yb)
t2 = time.time()
print("Eager step time:", t1 - t0, "TF.function step time:", t2 - t1)


model.fit final val_acc: 0.8116666674613953


Eager step time: 0.3880500793457031 TF.function step time: 0.08506584167480469


2026-04-25 13:31:06.113991: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-04-25 13:31:06.199057: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Final summary table

| Section | Result artifact | notes |
|---|---|---|
| Custom optimizer | `MyMomentumOptimizer` training behavior | Closely mirrors built-in SGD+momentum |
| Manual loop | per-epoch train/val metrics + tqdm logs | Full control over training internals |
| `@tf.function` | eager vs graph timing printout | Graph mode usually faster after warmup |
| `model.fit` baseline | final val accuracy | Should be within ~1% of manual loop |

Custom loops are most useful when you need non-standard objectives, logging, or update rules that are awkward to express through `compile`/`fit`.
